# Road condition assessment — Nakuru pilot

End-to-end pipeline, in order: pull imagery for a Nakuru AOI (GEE + Esri basemap +
Mapillary street-level images) &rarr; optionally fine-tune a road-damage detector on
your own labeled images &rarr; run detection &rarr; geolocate results onto the road
network &rarr; score road segments &rarr; compare against ground truth &rarr; export
for a road authority.

**Steps, in run order:**
1. Setup
2. Earth Engine — Nakuru County AOI + Sentinel-2 context layer
3. Road network — Nakuru city pilot area
4. Street-level imagery — Mapillary pull
5. *(Optional)* Fine-tune the detector on your own labeled set — best run in Colab
6. Load the detection model (fine-tuned from Step 5, or pretrained if you skipped it)
7. Run inference
8. Geolocate detections onto the road network
9. Score road segments
10. Visualize results
11. Ground-truth validation
12. Export outputs

**Before running:** get a Google Earth Engine project, a Mapillary access token, and
(if using Step 5) a Roboflow account for annotation — details are in Steps 1 and 5.
Steps 2–4 and 6–12 need only a normal Python environment; Step 5 assumes a Colab GPU
runtime but notes how to adapt it if you're training locally.

## Step 1: Setup

In [ ]:
# 1a. Install dependencies (uncomment on first run)
# %pip install earthengine-api geemap osmnx geopandas shapely folium requests ultralytics roboflow pandas numpy pillow pyyaml --quiet
print("Dependencies ready — uncomment the %pip line above on first run.")

In [ ]:
# 1b. Imports
import os
import time
import json
from pathlib import Path

import ee
import geemap
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import requests
from shapely.geometry import Point, LineString

pd.set_option("display.max_columns", 50)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:", IN_COLAB)

In [ ]:
# 1c. Config — fill these in
GEE_PROJECT_ID = "your-gee-project-id"          # https://code.earthengine.google.com/
MAPILLARY_TOKEN = "your-mapillary-access-token"  # https://www.mapillary.com/dashboard/developers

# Roboflow — only needed if you run Step 5 (fine-tuning)
ROBOFLOW_API_KEY = "your-roboflow-api-key"
ROBOFLOW_WORKSPACE = "your-workspace"
ROBOFLOW_PROJECT = "your-project-name"
ROBOFLOW_VERSION = 1

# Model weights — Step 6 falls back to downloading the pretrained checkpoint here
# if Step 5 wasn't run, or points at your fine-tuned best.pt if it was.
MODEL_WEIGHTS_PATH = "models/YOLOv8_Small_RDD.pt"

# AOI: Nakuru pilot
AOI_PLACE_CITY = "Nakuru, Kenya"       # OSM place name for the road-network pull (city-scale, keeps the pilot fast)
AOI_COUNTY_NAME = "Nakuru"             # FAO GAUL level-2 name for the broader satellite context layer

BASE_DIR = Path.cwd()
MODELS_DIR = BASE_DIR / "models"
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
for d in (MODELS_DIR, DATA_DIR, OUTPUT_DIR):
    d.mkdir(exist_ok=True)

## Step 2: Earth Engine — Nakuru County AOI + Sentinel-2 context layer

This gives a broad monitoring layer (Sentinel-2, NDVI/change along road buffers) at
the Nakuru **County** scale — useful for flagging unpaved-road erosion and vegetation
encroachment, not for pothole-level detail (satellite resolution isn't fine enough
for that — that's what Steps 4 and 7 are for).

In [ ]:
# 2a. Authenticate + initialize
ee.Authenticate()  # opens a browser flow the first time; cached afterwards
ee.Initialize(project=GEE_PROJECT_ID)

In [ ]:
# 2b. Nakuru County boundary from FAO GAUL
#
# NOTE: Kenya's GAUL *level 1* units are the pre-2010 provinces (e.g. "Rift Valley",
# "Nyanza"), not the 47 counties created by the 2010 constitution — so
# ADM1_NAME == "Nakuru" never matches. Nakuru exists as a *level 2* unit instead,
# so we query FAO/GAUL/2015/level2 and match on ADM2_NAME. ADM0_NAME narrows this
# to Kenya so we don't accidentally match a same-named unit elsewhere.
gaul2 = ee.FeatureCollection("FAO/GAUL/2015/level2")
nakuru_county = gaul2.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "Kenya"),
        ee.Filter.eq("ADM2_NAME", AOI_COUNTY_NAME),
    )
)

n_matches = nakuru_county.size().getInfo()
if n_matches == 0:
    # Self-diagnosing fallback: list the actual Kenyan ADM2 names so a naming
    # mismatch is obvious immediately instead of guessing.
    available = sorted(
        gaul2.filter(ee.Filter.eq("ADM0_NAME", "Kenya")).aggregate_array("ADM2_NAME").getInfo()
    )
    raise ValueError(
        f"No FAO GAUL level-2 unit found for Kenya / '{AOI_COUNTY_NAME}'. "
        f"Available ADM2 names include: {available}"
    )

aoi_county = nakuru_county.geometry()
print(f"Nakuru County AOI loaded ({n_matches} matching unit(s)).")

In [ ]:
# 2c. Pull a recent, low-cloud Sentinel-2 composite over the county
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi_county)
    .filterDate("2025-01-01", "2026-06-30")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    .median()
    .clip(aoi_county)
)

ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")

m_gee = geemap.Map(center=[-0.303, 36.080], zoom=10)
m_gee.addLayer(s2, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "Sentinel-2 RGB")
m_gee.addLayer(ndvi, {"min": -0.2, "max": 0.8, "palette": ["brown", "yellow", "green"]}, "NDVI")
m_gee.addLayer(aoi_county, {}, "Nakuru County boundary", opacity=0.15)
m_gee

## Step 3: Road network — Nakuru city pilot area

Street-level imagery and pothole detection only make sense at a finer scale than the
whole county, so the working pilot area is Nakuru city/town. Swap `AOI_PLACE_CITY`
for a smaller bounding box (`ox.graph_from_bbox`) if this is still too large to run
comfortably.

In [ ]:
# 3a. Pull the drivable road network for the pilot area
road_graph = ox.graph_from_place(AOI_PLACE_CITY, network_type="drive", simplify=True)
roads_gdf = ox.graph_to_gdfs(road_graph, nodes=False, edges=True).reset_index()
roads_gdf = roads_gdf[["osmid", "name", "highway", "geometry"]].copy()
roads_gdf["segment_id"] = roads_gdf.index
print(f"Pulled {len(roads_gdf)} road segments for {AOI_PLACE_CITY}")
roads_gdf.head()

In [ ]:
# 3b. Quick look at the network on an Esri World Imagery basemap (no API key needed)
bounds = roads_gdf.total_bounds  # minx, miny, maxx, maxy
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

m = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Esri World Imagery",
).add_to(m)
folium.GeoJson(roads_gdf, name="Road network", style_function=lambda x: {"color": "#00BFFF", "weight": 2}).add_to(m)
folium.LayerControl().add_to(m)
m

## Step 4: Street-level imagery — Mapillary

Pull images along the road network from the Mapillary Graph API. Potholes and cracks
are only reliably visible from street level (or drone), not from satellite imagery,
so this is the main source feeding the detection model.

In [ ]:
# 4a. Query Mapillary images within a bounding box
def get_mapillary_images(bbox, token, limit=200):
    # bbox = (west, south, east, north). Returns a list of image records.
    url = "https://graph.mapillary.com/images"
    params = {
        "access_token": token,
        "bbox": ",".join(str(b) for b in bbox),
        "fields": "id,geometry,compass_angle,captured_at,thumb_1024_url",
        "limit": limit,
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json().get("data", [])

mly_bbox = tuple(roads_gdf.total_bounds)  # (west, south, east, north)
mly_images = get_mapillary_images(mly_bbox, MAPILLARY_TOKEN)
print(f"Found {len(mly_images)} Mapillary images in the pilot AOI")

mly_df = pd.json_normalize(mly_images)
mly_df.head()

In [ ]:
# 4b. Download a working sample of images
IMG_DIR = DATA_DIR / "mapillary_images"
IMG_DIR.mkdir(exist_ok=True)

SAMPLE_SIZE = 100  # keep this modest for a pilot run
sample_df = mly_df.sample(min(SAMPLE_SIZE, len(mly_df)), random_state=42).reset_index(drop=True)

def download_image(url, out_path):
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    out_path.write_bytes(r.content)

local_paths = []
for i, row in sample_df.iterrows():
    out_path = IMG_DIR / f"{row['id']}.jpg"
    if not out_path.exists():
        download_image(row["thumb_1024_url"], out_path)
        time.sleep(0.1)  # be polite to the API
    local_paths.append(out_path)

sample_df["local_path"] = local_paths
print(f"Downloaded {len(sample_df)} images to {IMG_DIR}")

## Step 5 (optional): Fine-tune the detector on your own labeled set

Starts from the pretrained `YOLOv8_Small_RDD.pt` checkpoint (trained on the
CRDDC2022/RDD2022 dataset — see `github.com/oracl4/RoadDamageDetection`) and
fine-tunes it on a small hand-labeled set of Kenyan road images.

**Skip this whole step** if you're happy using the pretrained weights as-is — go
straight to Step 6, which falls back to downloading them automatically.

Best run with a Colab GPU runtime (Runtime &rarr; Change runtime type &rarr; T4 GPU);
`IN_COLAB` (set in Step 1) is used below to adapt paths if you're running locally
with your own GPU instead.

Before running: annotate a set of road images (bounding boxes around
potholes/cracks) — Roboflow's free tier is the fastest path, and its export step
auto-generates the snippet used below. Keep the same four classes the pretrained
model already knows — `Longitudinal Crack`, `Transverse Crack`, `Alligator Crack`,
`Potholes` — unless you have a specific reason to add one; matching classes gets the
most benefit from transfer learning. Aim for at least ~100-150 labeled images if you
can; fewer will still train, but expect a rougher first pass.

In [ ]:
# 5a. Check for a GPU (informational — proceeds either way)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'No GPU detected — fine-tuning will be slow on CPU.'

In [ ]:
# 5b. Set the working project directory (Drive if in Colab, local folder otherwise)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/road-condition-nakuru")
else:
    PROJECT_DIR = BASE_DIR / "training_run"
    print("Not running in Colab — using a local folder instead of Google Drive.")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print("Fine-tuning project files will be kept in:", PROJECT_DIR)

In [ ]:
# 5c. Install fine-tuning dependencies and pull the labeled dataset from Roboflow
%pip install ultralytics roboflow --quiet

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
rf_project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = rf_project.version(ROBOFLOW_VERSION).download("yolov8", location=str(PROJECT_DIR / "dataset"))

DATASET_DIR = dataset.location
DATA_YAML = f"{DATASET_DIR}/data.yaml"
print("Dataset downloaded to:", DATASET_DIR)

In [ ]:
# 5d. Sanity-check the dataset config and class list
import yaml

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

print(data_cfg)
print(f"\n{data_cfg['nc']} classes: {data_cfg['names']}")

In [ ]:
# 5e. Get the pretrained RDD2022 weights to fine-tune from (skip if already present)
pretrained_path = PROJECT_DIR / "YOLOv8_Small_RDD.pt"
if not pretrained_path.exists():
    url = "https://raw.githubusercontent.com/oracl4/RoadDamageDetection/main/models/YOLOv8_Small_RDD.pt"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    pretrained_path.write_bytes(r.content)

from ultralytics import YOLO

finetune_base_model = YOLO(str(pretrained_path))
print(finetune_base_model.names)  # confirm this matches (or is a superset of) your new dataset's classes

Key choices for a **small** dataset, used in the training call below:
- `freeze=10` — locks the first 10 backbone layers so training only adapts the
  later/head layers; this is what small-dataset transfer learning wants, since it
  avoids overwriting the general crack/pothole features already learned from RDD2022.
- Low `lr0` — small, careful updates rather than the larger learning rate you'd use
  training from scratch.
- `patience` — stops early if validation performance stalls, which matters more
  than usual when overfitting risk is high on a small set.
- Modest `epochs` — start around 50-100 and watch the validation curves; a small
  dataset overfits fast, so more epochs isn't automatically better here.

In [ ]:
# 5f. Fine-tune
finetune_results = finetune_base_model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=16,
    device=0 if IN_COLAB else "cpu",
    freeze=10,
    lr0=0.001,
    patience=15,
    project=str(PROJECT_DIR / "runs"),
    name="nakuru_finetune",
)

run_dir = PROJECT_DIR / "runs" / "nakuru_finetune"
print("Training run saved to:", run_dir)

In [ ]:
# 5g. Validate and print metrics (mAP, precision, recall per class)
finetune_metrics = finetune_base_model.val()
print(finetune_metrics.box.map)     # mAP50-95
print(finetune_metrics.box.map50)   # mAP50
print(finetune_metrics.box.maps)    # per-class mAP50-95

In [ ]:
# 5h. Show the training curves, confusion matrix, and PR curve Ultralytics saved
from IPython.display import Image, display

for fname in ["results.png", "confusion_matrix.png", "BoxPR_curve.png"]:
    fig_path = run_dir / fname
    if fig_path.exists():
        print(fname)
        display(Image(filename=str(fig_path), width=600))

In [ ]:
# 5i. Visual sanity check on a few validation images
import glob

val_images = glob.glob(f"{DATASET_DIR}/valid/images/*")[:6]
best_weights = run_dir / "weights" / "best.pt"
finetuned_model = YOLO(str(best_weights))

for i, img_path in enumerate(val_images):
    result = finetuned_model.predict(source=img_path, conf=0.25, verbose=False)[0]
    out_path = f"/tmp/pred_{i}.jpg"
    result.save(filename=out_path)
    display(Image(filename=out_path, width=500))

In [ ]:
# 5j. Point the rest of the pipeline at the fine-tuned weights
MODEL_WEIGHTS_PATH = str(best_weights)
print("MODEL_WEIGHTS_PATH updated to the fine-tuned checkpoint:")
print(MODEL_WEIGHTS_PATH)

## Step 6: Load the detection model

Uses the fine-tuned weights from Step 5 if you ran it; otherwise falls back to
downloading the pretrained RDD2022 checkpoint automatically.

In [ ]:
# 6a. Load whichever weights MODEL_WEIGHTS_PATH currently points at
from ultralytics import YOLO

weights_path = Path(MODEL_WEIGHTS_PATH)
if not weights_path.exists():
    weights_path = MODELS_DIR / "YOLOv8_Small_RDD.pt"
    if not weights_path.exists():
        print("No local weights found — downloading the pretrained RDD2022 checkpoint...")
        url = "https://raw.githubusercontent.com/oracl4/RoadDamageDetection/main/models/YOLOv8_Small_RDD.pt"
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        weights_path.write_bytes(r.content)
    MODEL_WEIGHTS_PATH = str(weights_path)

model = YOLO(MODEL_WEIGHTS_PATH)
print("Loaded model from:", MODEL_WEIGHTS_PATH)
print(model.names)

## Step 7: Run inference over the pulled images

Rather than training from scratch, this uses whichever pretrained/fine-tuned
detector Step 6 loaded:

- **RDD2022 / CRDDC weights** (the default) — trained on six countries' road
  imagery; classes cover longitudinal cracks, transverse cracks, alligator
  cracking, and potholes.
- **Your fine-tuned weights** from Step 5, if you ran it — adapted to Kenyan road
  imagery specifically.

In [ ]:
# 7a. Run inference over the downloaded Mapillary images
CONF_THRESHOLD = 0.25

detection_rows = []
for i, row in sample_df.iterrows():
    result = model.predict(source=str(row["local_path"]), conf=CONF_THRESHOLD, verbose=False)[0]
    for box in result.boxes:
        cls_id = int(box.cls.item())
        detection_rows.append(
            {
                "image_id": row["id"],
                "lon": row["geometry.coordinates"][0],
                "lat": row["geometry.coordinates"][1],
                "captured_at": row["captured_at"],
                "class_id": cls_id,
                "class_name": model.names[cls_id],
                "confidence": float(box.conf.item()),
            }
        )

detections_df = pd.DataFrame(detection_rows)
print(f"{len(detections_df)} detections across {sample_df.shape[0]} images")
detections_df.head()

## Step 8: Geolocate detections onto the road network

Each detection is placed at its source image's GPS location (a reasonable
approximation for a pilot — precise triangulation would need stereo/depth data),
then snapped to the nearest road segment.

In [ ]:
# 8a. Build a GeoDataFrame of detections and snap to the nearest road segment
detections_gdf = gpd.GeoDataFrame(
    detections_df,
    geometry=gpd.points_from_xy(detections_df["lon"], detections_df["lat"]),
    crs="EPSG:4326",
)

roads_gdf = roads_gdf.set_crs("EPSG:4326", allow_override=True)

# Project to a local metric CRS for accurate distance-based snapping (UTM zone 37S covers Nakuru)
metric_crs = "EPSG:32737"
detections_m = detections_gdf.to_crs(metric_crs)
roads_m = roads_gdf.to_crs(metric_crs)

joined = gpd.sjoin_nearest(detections_m, roads_m, how="left", distance_col="dist_to_road_m")
detections_gdf["segment_id"] = joined["segment_id"].values
detections_gdf["dist_to_road_m"] = joined["dist_to_road_m"].values

print(detections_gdf[["class_name", "confidence", "segment_id", "dist_to_road_m"]].head())

## Step 9: Score road segments

In [ ]:
# 9a. Aggregate a per-segment condition score
severity_weight = {
    "Longitudinal Crack": 1,
    "Transverse Crack": 1,
    "Alligator Crack": 2,
    "Potholes": 3,
}  # cracks lighter, potholes heavier — matches YOLOv8_Small_RDD.pt's class names; adjust if you fine-tuned with different classes

detections_gdf["weight"] = detections_gdf["class_name"].map(severity_weight).fillna(1)

segment_scores = (
    detections_gdf.groupby("segment_id")
    .agg(defect_count=("class_name", "count"), condition_score=("weight", "sum"))
    .reset_index()
)

roads_scored = roads_gdf.merge(segment_scores, on="segment_id", how="left")
roads_scored[["defect_count", "condition_score"]] = roads_scored[["defect_count", "condition_score"]].fillna(0)

# Simple condition class for reporting
bins = [-1, 0, 3, 8, np.inf]
labels = ["No data / clear", "Fair", "Poor", "Needs urgent maintenance"]
roads_scored["condition_class"] = pd.cut(roads_scored["condition_score"], bins=bins, labels=labels)

roads_scored[["segment_id", "name", "defect_count", "condition_score", "condition_class"]].sort_values(
    "condition_score", ascending=False
).head(10)

## Step 10: Visualize results

In [ ]:
m_result = folium.Map(location=center, zoom_start=13, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Esri World Imagery",
).add_to(m_result)

condition_colors = {
    "No data / clear": "#9e9e9e",
    "Fair": "#ffd54f",
    "Poor": "#ff8a65",
    "Needs urgent maintenance": "#e53935",
}

folium.GeoJson(
    roads_scored,
    name="Road condition",
    style_function=lambda f: {
        "color": condition_colors.get(f["properties"].get("condition_class"), "#9e9e9e"),
        "weight": 4,
    },
    tooltip=folium.GeoJsonTooltip(fields=["name", "defect_count", "condition_class"]),
).add_to(m_result)

for _, row in detections_gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color="black",
        fill=True,
        fill_opacity=0.7,
        popup=f"{row['class_name']} ({row['confidence']:.2f})",
    ).add_to(m_result)

folium.LayerControl().add_to(m_result)
m_result

## Step 11: Ground-truth validation

Compare model detections against a manually verified survey — even a simple CSV of
`lat, lon, defect_type` collected by driving/walking the pilot segment works for a
first pass (a template is included at `data/ground_truth_template.csv`). This is the
step that lets you report a real precision/recall figure, rather than just a raw
count of detections.

In [ ]:
# 11a. Load ground truth and compute precision/recall against detections
GROUND_TRUTH_PATH = DATA_DIR / "ground_truth_nakuru.csv"  # columns: lat, lon, defect_type

if GROUND_TRUTH_PATH.exists():
    gt_df = pd.read_csv(GROUND_TRUTH_PATH)
    gt_gdf = gpd.GeoDataFrame(
        gt_df, geometry=gpd.points_from_xy(gt_df["lon"], gt_df["lat"]), crs="EPSG:4326"
    ).to_crs(metric_crs)

    MATCH_RADIUS_M = 15  # how close a detection needs to be to a ground-truth point to count as a match
    matched = gpd.sjoin_nearest(gt_gdf, detections_m, how="left", distance_col="dist_m")
    matched["matched"] = matched["dist_m"] <= MATCH_RADIUS_M

    recall = matched["matched"].mean()
    print(f"Recall vs ground truth (within {MATCH_RADIUS_M} m): {recall:.1%}")

    gt_matched_ids = set(matched.loc[matched["matched"], "index_right"])
    precision = len(gt_matched_ids) / len(detections_gdf) if len(detections_gdf) else float("nan")
    print(f"Precision (detections matched to a ground-truth point): {precision:.1%}")
else:
    print(f"No ground-truth file found at '{GROUND_TRUTH_PATH}'.")
    print("Fill in data/ground_truth_template.csv and save it as ground_truth_nakuru.csv to run validation.")

## Step 12: Export outputs

In [ ]:
roads_out = OUTPUT_DIR / "nakuru_road_condition.geojson"
detections_out = OUTPUT_DIR / "nakuru_detections.csv"

roads_scored.to_file(roads_out, driver="GeoJSON")
detections_gdf.drop(columns="geometry").assign(
    lon=detections_gdf.geometry.x, lat=detections_gdf.geometry.y
).to_csv(detections_out, index=False)

print(f"Wrote {roads_out}")
print(f"Wrote {detections_out}")

## Next steps

- Swap the pilot bounding box for the specific Nakuru road segment(s) you want to
  headline in the GEOSK submission.
- If Mapillary coverage is thin in parts of Nakuru, fill gaps with your own
  dashcam/phone footage — same download-and-infer flow applies once you have local
  images with GPS EXIF or a matching coordinates log.
- If you get access to a drone flight over a short segment, run the same detection
  cell over the orthomosaic tiles for a higher-fidelity demo case.
- If Step 5's fine-tune shows weak mAP on the crack subclasses specifically,
  collapsing Longitudinal/Transverse/Alligator into one "Crack" class (keeping
  "Potholes" separate) will likely outperform forcing the finer distinction on a
  small dataset.